# T-SOHO Phase 4 (Kaggle)
This notebook calls repository source only; it contains no model implementation.

In [ ]:
# User-editable configuration
REPO_DIR = '/kaggle/input/soho-cl'
CIFAR_ROOT = '/kaggle/input/cifar100'
CHECKPOINT_PATH = '/kaggle/input/vit-checkpoint/model.safetensors'
CACHE_DIR = '/kaggle/working/feature_cache'
OUTPUT_DIR = '/kaggle/working/outputs'
SEED = 1993
NUM_TASKS = 10
RANKS = [8, 16, 32, 64]
RIDGE_LAMBDA = 1.0
METHODS = ['raw_ridge','random_orthogonal_code','truncated_simplex_code','spectral_confusion_code']


In [ ]:
import os, shutil, sys, torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
WORK_REPO = '/kaggle/working/SOHO-CL'
if not os.path.exists(WORK_REPO): shutil.copytree(REPO_DIR, WORK_REPO, ignore=shutil.ignore_patterns('.git','data','outputs','feature_cache'))
os.chdir(WORK_REPO)
!pip -q install -r requirements-kaggle.txt


In [ ]:
# Verify checkpoint and source tests/preflight
!python -m pytest -q tests/test_tsoho_learner.py tests/test_tsoho_math.py tests/test_backbone_checkpoint.py
!python tools/checkpoint_preflight.py --root {CIFAR_ROOT} --checkpoint {CHECKPOINT_PATH} --checkpoint-size 346284714 --checkpoint-sha256 32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b --seed {SEED}


In [ ]:
# Extract cache once. This writes only /kaggle/working cache, never learner state.
if not os.path.exists(os.path.join(CACHE_DIR, 'metadata.json')):
    cmd = f'python tools/experiment_runner.py --extract-features-only --root {CIFAR_ROOT} --backbone-checkpoint {CHECKPOINT_PATH} --backbone-checkpoint-size 346284714 --backbone-checkpoint-sha256 32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b --feature-cache-dir {CACHE_DIR} --output-dir {OUTPUT_DIR}/cache_extract --dataset CIFAR-100 --model-name vit_base_patch16_224 --data-augmentation vit --seed {SEED} --num-tasks {NUM_TASKS} --num-classes 100 --device cuda --batch-size 128 --num-workers 8'
    assert os.system(cmd) == 0, cmd
else:
    print('Using existing cache:', CACHE_DIR)


In [ ]:
# Raw Ridge runs once; code methods run once per rank using the same cache/order/lambda.
for method in METHODS:
    ranks = [None] if method == 'raw_ridge' else RANKS
    for rank in ranks:
        tag = method if rank is None else f'{method}_r{rank}'
        suffix = '' if rank is None else f' --rank {rank}'
        cmd = f'python tools/experiment_runner.py --method {method}{suffix} --ridge-lambda {RIDGE_LAMBDA} --feature-cache-dir {CACHE_DIR} --output-dir {OUTPUT_DIR}/{tag} --dataset CIFAR-100 --model-name vit_base_patch16_224 --num-classes 100 --num-tasks {NUM_TASKS} --seed {SEED} --device cuda --resume'
        assert os.system(cmd) == 0, cmd


In [ ]:
# Aggregate/plot artifacts after runs, then zip for download
!cd /kaggle/working && zip -r phase4_artifacts.zip outputs
